In [9]:
#路径包，操作路径
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
#显示,和print的区别,显示表格
from IPython.display import display

#循环显示路径，保证能在这些路径下打开文件
candidates = [Path.cwd() / "nba_player_seasons.csv"] + [
    p / "data/slide03/nba_player_seasons.csv"
    for p in [Path.cwd(), *Path.cwd().parents]
]
#如果路径下有文件，则取出来，否则报错
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Open inside the course repository or upload nba_player_seasons.csv here.")
nba = pd.read_csv(data_path)
print(f"Loaded {len(nba):,} rows and {len(nba.columns)} columns")
display(nba.head())
#将类型形成df，命名
display(nba.dtypes.rename("dtype").to_frame())
#nan的个数
display(nba.isna().sum().rename("missing").to_frame())
#两个字段里面有没有重复的，any表示true(非个数)，assert为报错，是false就报错，去重 只要出现一个 True，any() 就返回 True ,全部为 False 才返回 False
assert not nba.duplicated(["player_id", "season"]).any()
#size是数数，统计个数（一共有多少个元素）
display(nba.groupby("season").size().rename("player_seasons").to_frame())
#统计个数，不包含na（每个值各出现了几次）
display(nba["position"].value_counts(dropna=False).to_frame())

Loaded 5,386 rows and 13 columns


,season,player_id,player,position,team_id,games,minutes_per_game,fg_pct,ft_pct,rebounds,assists,turnovers,points
0,2015-16,201939,Stephen Curry,PG,GSW,79,34.2,0.504,0.908,5.4,6.7,3.3,30.1
1,2015-16,201935,James Harden,SG,HOU,82,38.1,0.439,0.860,6.1,7.5,4.6,29.0
2,2015-16,201142,Kevin Durant,SF,OKC,72,35.8,0.505,0.898,8.2,5.0,3.5,28.2
3,2015-16,202326,DeMarcus Cousins,C,SAC,65,34.6,0.451,0.718,11.5,3.3,3.8,26.9
4,2015-16,2544,LeBron James,SF,CLE,76,35.6,0.520,0.731,7.4,6.8,3.3,25.3


,dtype
season,str
player_id,int64
player,str
position,str
team_id,str
games,int64
minutes_per_game,float64
fg_pct,float64
ft_pct,float64
rebounds,float64


,missing
season,0
player_id,0
player,0
position,26
team_id,0
games,0
minutes_per_game,0
fg_pct,0
ft_pct,0
rebounds,0


,player_seasons
season,
2015-16,476
2016-17,486
2017-18,540
2018-19,530
2019-20,529
2020-21,540
2021-22,605
2022-23,539
2023-24,572


,count
position,
SG,1277
PF,1069
PG,1020
C,1010
SF,984
NaN,26


In [10]:
MIN_GAMES = 40
MIN_MINUTES = 15
#select查询 copy阻断，防止篡改原文
eligible = nba.loc[
    (nba["games"] >= MIN_GAMES)
    & (nba["minutes_per_game"] >= MIN_MINUTES)
].copy()
#去重排序，取唯一值
all_seasons = sorted(nba["season"].unique())
#reindex，比较all_seasons和eligible数据，如果有值,size判断每堆有几行
#数一下每个赛季各有多少条合格记录，并且把所有赛季都列出来，一个赛季都没有的就写 0。
#reindex对比
counts = eligible.groupby("season").size().reindex(all_seasons, fill_value=0)
display(counts.rename("eligible_player_seasons").to_frame())


,eligible_player_seasons
season,
2015-16,284
2016-17,294
2017-18,285
2018-19,299
2019-20,277
2020-21,292
2021-22,308
2022-23,298
2023-24,288


In [11]:
# TODO: Apply the season/position restrictions for your claim.
# TODO: Calculate your metric and sample sizes.
# TODO: Assign the final table to evidence and display(evidence).
analysis = eligible.copy()
analysis["high_scorer"] = analysis["points"] >= 20
display(analysis)
evidence = (
    analysis.groupby("season")
    #聚合
    .agg(
        eligible_players=("player_id", "size"),
        #true和false
        high_scorers=("high_scorer", "sum")
    )
    .reindex(all_seasons)
)

display(evidence)

evidence["high_scorer_pct"] = (
    evidence["high_scorers"]
    / evidence["eligible_players"]
    * 100
).round(2)

display(evidence)


,season,player_id,player,position,team_id,games,minutes_per_game,fg_pct,ft_pct,rebounds,assists,turnovers,points,high_scorer
0,2015-16,201939,Stephen Curry,PG,GSW,79,34.2,0.504,0.908,5.4,6.7,3.3,30.1,True
1,2015-16,201935,James Harden,SG,HOU,82,38.1,0.439,0.860,6.1,7.5,4.6,29.0,True
2,2015-16,201142,Kevin Durant,SF,OKC,72,35.8,0.505,0.898,8.2,5.0,3.5,28.2,True
3,2015-16,202326,DeMarcus Cousins,C,SAC,65,34.6,0.451,0.718,11.5,3.3,3.8,26.9,True
4,2015-16,2544,LeBron James,SF,CLE,76,35.6,0.520,0.731,7.4,6.8,3.3,25.3,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5230,2024-25,1630556,Kessler Edwards,SF,DAL,40,15.2,0.496,0.923,2.9,1.1,0.4,4.2,False
5231,2024-25,1631200,Kris Murray,SF,POR,69,15.1,0.419,0.456,2.6,1.0,0.6,4.2,False
5232,2024-25,1642345,Oso Ighodaro,PF,PHX,61,17.1,0.604,0.580,3.6,1.2,0.6,4.2,False
5246,2024-25,201587,Nicolas Batum,PF,LAC,78,17.5,0.437,0.810,2.8,1.1,0.4,4.0,False


,eligible_players,high_scorers
season,,
2015-16,284,20
2016-17,294,32
2017-18,285,27
2018-19,299,33
2019-20,277,31
2020-21,292,40
2021-22,308,36
2022-23,298,54
2023-24,288,47


,eligible_players,high_scorers,high_scorer_pct
season,,,
2015-16,284,20,7.04
2016-17,294,32,10.88
2017-18,285,27,9.47
2018-19,299,33,11.04
2019-20,277,31,11.19
2020-21,292,40,13.70
2021-22,308,36,11.69
2022-23,298,54,18.12
2023-24,288,47,16.32
